In [4]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [5]:
RANDOM_STATE = 22

# Preprocessing

Decisions from [EDA.ipynb](EDA.ipynb)

- 394 rows have missing values. Use training medians for missing numbers and `Missing` for missing diet labels.
- The audit found zero duplicates and zero invalid values. Keep all rows, including the 37 screen-time flags and the one low-attendance flag.
- Use `Mental_Health_Risk` as the target and leave `Student_ID` out of the features. Use a stratified 80/20 split to keep the class proportions.


In [ ]:
path = Path("data/raw/student-mental.csv")
if not path.is_file():
    path = Path("..") / path

df = pd.read_csv(path)
target = "Mental_Health_Risk"
X = df.drop(columns=["Student_ID", target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

In [ ]:
number_columns = X_train.select_dtypes(include="number").columns
category_columns = X_train.select_dtypes(exclude="number").columns

number_steps = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)

category_steps = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="Missing"),
    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
)

preprocessor = ColumnTransformer([
    ("numeric", number_steps, number_columns),
    ("categorical", category_steps, category_columns),
]).set_output(transform="pandas")

In [ ]:
# Learn from the training set. Apply the same steps to the test set.
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)